# FDM modeling of ARK-cross sections

We use the developed code from "src/ARK_geotop.py"

In [184]:
import os
import sys
from typing import Any
from glob import glob
from pprint import pprint
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle, Patch
import pickle

from tools.fdm.src.mfgrid import Grid
from tools.fdm.src.fdm3Blom import Fdm3
from tools.etc.etc import logo

from mf6lab.Projects.ARK_RWS.src.ARK_geotop import Dirs


print(sys.executable)

# --- Needed to make figure separate from the notebook and interactive
%matplotlib qt

# --- Notebook name for logo
NOTEBOOK_NAME = "ARK_fdm.ipynb"
# --- Seet the namespace for the relevant directories
dirs = Dirs()

# --- Get the paths and names of  the geotop pdf files in the order they are in dirs.dino
xsec_paths = {i:name for i, name in enumerate(glob(dirs.dino + '*.pdf'))}
xsec_names = {i:os.path.basename(name) for i, name in enumerate(glob(dirs.dino + '*.pdf'))}

# --- Pickling
def pickleto(var:Any, basename:str, parent:str=dirs.data):
    """Pickle var to os.path.join(dirs.data, basename)"""
    if not basename.endswith('.pkl'):
        basename += ".pkl"

    pkl_file = os.path.join(parent, basename)
    with open(pkl_file, 'wb') as f:
        print(f"Pickled {basename} --> {parent}")        
        pickle.dump(var, f)

# --- Unpickling
def picklefrom(basename:str, parent:str=dirs.data)->Any:
    """Unpickle varname from os.path.join(parent, basename)"""
    if not basename.endswith(".pkl"):
        basename += ".pkl"
          
    pkl_file = os.path.join(parent, basename)
    with open(pkl_file, 'rb') as f:
        print(f"Loaded {basename} <-- {parent}")        
        return pickle.load(f)


def spy(idx_arr):
    """Show where the index labels are in the xsec idx_arr."""
    arr = np.squeeze(idx_arr)
        
    fig, ax = plt.subplots(figsize=(10, 6))
    if np.issubdtype(idx_arr.dtype, np.integer):
        title = "Location of legend indices in xsec.idx_arr"
        classes = np.unique(arr)
        cmap = plt.get_cmap('tab20', len(classes) - 1)
    else:
        title = "imshow of given array"
        cmap = plt.get_cmap('viridis')
    ax.set_title(title)
    mappable = ax.imshow(arr, cmap=cmap, origin='upper')
    fig.colorbar(mappable)
    plt.show()

def extent_patch(extent, **kwargs):
    """Return a rectangle patch object.
    
    Parameters
    ----------
    extent: np.array | tuple | list of 4 floats
        xmin, xmax, ymin, ymax
    kwargs: dict
        more parameter passed on to the Rectangle object.
    
    use it as
    as.add_patch(rect)
    """
    xmin, xmax, ymin, ymax = extent
    return Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, **kwargs)
    
def find_horizonal_intersections(cs, z0):
    """Return horizontal contourline intersections at z0.
    
    Parameters
    ----------
    cs: contour set (returned by plt.contour or ax.contour)
        Contour set with all contour information.
    z0: float
        z-location where contour labels are to be placed along horiontal.
    """
    points = []

    for path in cs.get_paths():
        v = path.vertices
        x, z = v[:, 0], v[:, 1]
        if len(x) < 2:
            continue
        dz0 = z - z0
        mask = dz0[:-1] * dz0[1:] <=0
        idz = np.where(mask)[0]
        if idz.size > 0:
            for i in idz:
                t = (z0 - z[i]) / (z[i+1] - z[i] + 1e-12)
                xi = x[i] + t * (x[i+1] - x[i])
                points.append((xi, z0))
    return points

def find_vertical_intersections(cs, x0):
    """Return vertical contourline intersections at x0.
    
    Parameters
    ----------
    cs: contour set (returned by plt.contour or ax.contour)
        Contour set with all contour information.
    x0: float
        x-location where contour labels are to be place along vertical.
    """
    points = []

    for path in cs.get_paths():
        v = path.vertices
        x, z = v[:, 0], v[:, 1]
        dx0 = x - x0
        mask = dx0[:-1] * dx0[1:] <=0
        idx = np.where(mask)[0]
        if idx.size > 0:
            i = idx[0]
            t = (x0 - x[i]) / (x[i+1] - x[i] + 1e-12)
            zi = z[i] + t * (z[i+1] - z[i])
            if zi < -45. or zi > -5.:
                continue
            points.append((x0, zi))
    return points

# --- Color for empty legend (empty voxel with geo_unit 'none')
WHITE_01 = np.array([1., 1., 1.])

/Users/Theo/Development/python/mf6_tools/mf6lab/.venv/bin/python


In [3]:
pprint(xsec_names)

{0: 'BRO GeoTOP Verticale doorsnede geologische eenheid 130173,479431.pdf',
 1: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse '
    '126311,474187.pdf',
 2: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse '
    '130049,479466.pdf',
 3: 'BRO GeoTOP Verticale doorsnede geologische eenheid 129926,479462.pdf',
 4: 'BRO GeoTOP Verticale doorsnede geologische eenheid 127484,477893.pdf',
 5: 'BRO GeoTOP Verticale doorsnede geologische eenheid 130049,479466.pdf',
 6: 'BRO GeoTOP Verticale doorsnede geologische eenheid 126311,474187.pdf',
 7: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse '
    '129926,479462.pdf',
 8: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse '
    '127484,477893.pdf',
 9: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse '
    '130173,479431.pdf'}


In [208]:
geotop_xsecs = picklefrom("geotop_xsecs.pkl")

Loaded geotop_xsecs.pkl <-- /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/


## Deal with the second cross section only

In [209]:
isec = 1
xsec = geotop_xsecs[xsec_names[isec]]

print(f"Dealing with xsec {isec}:\n{xsec.name}") 

# --- find the ARK it wronly has index 1 in row 6
ix_ARK = np.where(xsec.idx_arr[6] == 1)[0]

spy(xsec.idx_arr)


Dealing with xsec 1:
BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 126311,474187.pdf


/var/folders/90/m51x_b713y561gzh2kzy18d00000gq/T/ipykernel_1197/3658457952.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Repair the idx_arr for the ARK canal which looks now filled with material 'a'

The problem is that the idx_arr contains index 1 (anthropogenic) inside the ARK,
which was likely caused by some vertical grid line disturbing the color_match,
so the light gray legend color (1) was matched instead of white (0).

To find ARK look for index = 1 in row 6 to find ARK incision in the xsec.
Then replace all the 1 indices in that column with 0 to make it empty.

In [212]:
idx_arr = xsec.idx_arr.copy()

# --- The columns have idx 1 incorrectly
cols = np.where(idx_arr[1] == 1)[0]

# --- Replace by index 0 ('none')
for j in cols:
    rows = idx_arr[:, j] == 1
    idx_arr[rows, j] = 0
    
# --- Check
spy(idx_arr)

/var/folders/90/m51x_b713y561gzh2kzy18d00000gq/T/ipykernel_1197/3658457952.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


When this works replace the xsec.idx_arr with the corrected version

In [213]:
# --- Replace origional idx_arr by corrected one
xsec.idx_arr = idx_arr

# --- Check
spy(xsec.idx_arr)
print('idx_arr repaired')

idx_arr repaired


/var/folders/90/m51x_b713y561gzh2kzy18d00000gq/T/ipykernel_1197/3658457952.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Model grid

In [231]:
def set_ARK_xsec(xsec):
    # --- Basic properties
    xsec.ground_elev = -1.3
    xsec.stage  = -0.4  # --- Water level of ARK    
    xsec.d_damw = 0.5   # --- Thickness of sheet piling
    xsec.z_damw = -12.  # --- Bottom elevation of sheet piling
    xsec.hpp    = xsec.ground_elev - 1 # --- Water level in ditches
    xsec.c_drainage = 100. # --- Areal drainage resistance (phi - hpp) = Nc about 0.1 m
    
    # --- Add legend color and rhow (wet density) to geo_units
    for (_, gu), color in zip(xsec.geo_units.items(), xsec.leg_colors[1:]):
        gu['color'] = color
        gu['rhow'] = gu['n'] * 1000 + (1 - gu['n']) * gu['rho']
            
    # --- where in the cross section is the ARK
    ixARK = np.where(xsec.idx_arr[10] == 0)[0][0]
    izARK = np.where(xsec.idx_arr[:, ixARK] == 0)[0]
    xsec.zbot = np.round(xsec.zm[izARK[-1]] - xsec.dz / 2, 1)
    # --- ARK water extent
    ae = np.round(np.array([xsec.x[ixARK], xsec.x[ixARK+1], xsec.zbot, xsec.stage]), 1)
    
    # --- Half width of ARK:
    xsec.b = (ae[1] - ae[0]) / 2

    # --- Centralize xsec.x around xARKmid
    xsec.xARKmid_orig     = 0.5 * (ae[0] + ae[1])
    t = (xsec.xARKmid_orig - xsec.x[0]) / (xsec.x[-1] - xsec.x[0])
    xsec.xy_hart_ARK=np.round(xsec.xyRD[0] + t * np.diff(xsec.xyRD, axis=0))
    
    # --- This causes all coordinates in xsec to be zeroed at xARKmid_orig
    xsec.world_extent_orig = xsec.world_extent
    xsec.world_extent[:2] -= xsec.xARKmid_orig
    
    # --- Grid x-coordinates (parallel to xsec) with 0 at xsec.xARK_orig
    # --- refined around the edges of the canal xsec.  
    slog = np.logspace(0, 2, 10)

    # --- b as shorthand of xsec.b
    b = xsec.b

    # --- x coordinates from the heart line of the ARK to the right (x>0)
    x_ = np.hstack((
        (b - np.logspace(0, np.log10(b), 10))[::-1],    # --- inside ARK, right of middle
        b, b + xsec.d_damw,                             # --- sheet piling
        b + slog[slog > xsec.d_damw],                   # --- increasing cell wirdth first 100 m 
        np.linspace(0, 2000, 21).clip(200, None)        # --- beyond this to 2000 m 100 m cells
    ))

    # --- For less clutter, round values
    x_ = np.round(x_, 1)

    # --- Mirror around heart line of ARK and remove doubles   
    x = np.unique(np.hstack((-x_[::-1], x_)))
    
    # --- Get the grid and attach all grid properties to it
    # --- The grid use the new x and the old z
    gr = Grid(x, None, xsec.z[xsec.z <= xsec.ground_elev])
    
    # --- ARK-extent --> mask_ARK
    gr.mask_ARK = gr.inblock(xx=(-b, b), yy=None, zz=(xsec.zbot, xsec.stage))
    gr.ARK_extent = np.array([-b, b, xsec.zbot, xsec.stage])
    
    # --- extent of the sheet piling left and right along canal      
    gr.mask_damwL = gr.inblock(xx=(-xsec.b - xsec.d_damw, -xsec.b), zz=(xsec.z_damw, 0))
    gr.mask_damwR = gr.inblock(xx=(+xsec.b, +xsec.b + xsec.d_damw), zz=(xsec.z_damw, 0))
    gr.damwL_extent = np.array([-b, -b - xsec.d_damw, xsec.z_damw, 0])
    gr.damwR_extent = np.array([b, b + xsec.d_damw, xsec.z_damw, 0])

    # --- Drainage: We use DRN as top boundary condition. There is neither evaporation        
    # --- nor recharge, just seepage from the ARK
    # --- Drains are in the cells with (hpp - 0.25 <= zm <= hpp + 0.25) and outside ARK
    gr.mask_DRN = np.logical_and(
        gr.inblock(xx=(gr.x[0], gr.x[-1]), zz=(xsec.hpp - 0.25, xsec.hpp + 0.25)),
        ~gr.mask_ARK
        )

    # --- DRN cell Id's 
    Idrn = gr.NOD[gr.mask_DRN]

    # --- DRN boundary condition
    DRN = np.zeros(len(Idrn), dtype=Fdm3.dtypes['drn'])
    DRN['Ig'] = Idrn
    DRN['C'] = xsec.c_drainage
    DRN['h'] = xsec.hpp
    gr.DRN = DRN

    # --- Other gr Arrays
    IBOUND = gr.const(1, dtype=int)
    IBOUND[gr.mask_damwL] =  0 # --- Inactive
    IBOUND[gr.mask_damwR] =  0 # --- Inactive
    IBOUND[gr.mask_ARK]   = -1 # --- Fixed head
    
    gr.IBOUND = IBOUND
    
    # --- No fixed Q
    gr.FQ = gr.const(0.)
    
    # --- No ohter fixed heads
    gr.FH = None
    
    # --- Initial heads hpp except insice ARK
    HI = gr.const(xsec.hpp)
    HI[gr.mask_ARK] = xsec.stage
    gr.HI = HI
    
    # --- Get legend index for all model cells
    gr.idx_arr = xsec.overlap(gr)
    
    # --- Get the hydraulic properties for the grid
    # --- Shape of props arrays are equal to gr.shape
    props  = xsec.get_props(idx_arr=gr.idx_arr)
    
    # --- Make the grid propertie arrays 3D
    for var in props:
        props[var] = props[var][:, np.newaxis, :]
    
    # --- The ARK-water body --> High vertical conductivity    
    props['kv'][gr.mask_ARK] = 1000.
    # --- The density of water.
    props['rho_wet'][gr.mask_ARK] = 2000.  

    # --- Add gr property arrays to grid object for later reference
    gr.kh = props['kh']
    gr.kv = props['kv']
    gr.K   = (gr.kh, gr.kh, gr.kv)
    gr.n   = props['n']
    gr.rho = props['rho']
    gr.rhow = props['rho_wet']

    # --- Don't really need this because the problem is steady state
    # --- However, the steady-state solution is appoached transiently for stability reasons.
    gr.S = gr.const(0.001)
    
    # --- Resistance at bottom of ARK
    iARK = np.where(gr.mask_ARK[0, 0, :])[0]
    izARK= np.where(gr.mask_ARK[:, 0, iARK[0]])[0][-1]
    gr.mask_ARK_bot = gr.const(0, dtype=bool)
    gr.mask_ARK_bot[izARK, 0, iARK] = True
    
    xsec.cBot = 0.
    if xsec.cBot > 0:
        gr.kv[gr.mask_ARK_bot] = 0.5 * gr.DZ[gr.mask_ARK_bot] / xsec.cBot
    
    # --- Add gr object to xsec. So that xsec carries all that is necessary to model it
    xsec.gr = gr
    

# --- Set all the above computed properties (by reference)
set_ARK_xsec(xsec)  

gr = xsec.gr

# --- Instantiate the model
mdl = Fdm3(gr=xsec.gr, K=gr.K, c=None, S=None, IBOUND=gr.IBOUND, HI=gr.HI, FQ=gr.FQ)

# --- Run the model
out = mdl.simulate(DRN=gr.DRN, RIV=None, GHB=None, FDR=None, tm=None, htol=1e-7, maxiter=50, verbose=False)

# --- Compute the stream function from out['Qx]
psi = gr.psi_row(out['Qx'])

/Users/Theo/Development/python/hydro_tools/tools/fdm/src/fdm3Blom.py:234: RuntimeWarning: divide by zero encountered in divide
  Rx2 = 0.5 * dx / (dy * dz) / kx
/Users/Theo/Development/python/hydro_tools/tools/fdm/src/fdm3Blom.py:235: RuntimeWarning: divide by zero encountered in divide
  Rx1 = 0.5 * dx / (dy * dz) / kx
/Users/Theo/Development/python/hydro_tools/tools/fdm/src/fdm3Blom.py:236: RuntimeWarning: divide by zero encountered in divide
  Ry  = 0.5 * dy / (dz * dx) / ky
/Users/Theo/Development/python/hydro_tools/tools/fdm/src/fdm3Blom.py:237: RuntimeWarning: divide by zero encountered in divide
  Rz  = 0.5 * dz / (dx * dy) / kz


Non-linear options: ['DRN'], starting outer iterations:
iouter =    0, err =        1.9 m, errBalance =    0.57472
iouter =    1, err =   0.057989 m, errBalance =    0.10477
iouter =    2, err =  0.0091526 m, errBalance =    0.03406
iouter =    3, err = 0.00093167 m, errBalance =   0.020857
iouter =    4, err =  0.0003925 m, errBalance =    0.01514
iouter =    5, err = 0.00016969 m, errBalance =  0.0072431
iouter =    6, err = 7.2543e-05 m, errBalance =  0.0020983
iouter =    7, err = 3.1155e-05 m, errBalance = 0.00030152
iouter =    8, err = 1.3351e-05 m, errBalance = 2.9524e-05
iouter =    9, err = 5.7252e-06 m, errBalance = -1.1345e-06
iouter =   10, err = 2.4539e-06 m, errBalance = 8.8195e-07
iouter =   11, err = 1.0519e-06 m, errBalance = -3.5464e-07
iouter =   12, err =  4.508e-07 m, errBalance = 1.4581e-07
iouter =   13, err = 1.9319e-07 m, errBalance = -6.0027e-08
iouter =   14, err = 8.2785e-08 m, errBalance = 2.4796e-08
Converged, normal termination.

===== Water balance of t

/Users/Theo/Development/python/hydro_tools/tools/fdm/src/mfgrid.py:2019: UserWarning: This only works when all flow is within plane zx given by row
  warnings.warn("This only works when all flow is within plane zx given by row")


# Plot the cross section

In [232]:
# --- Setup the figure. Adjust width to get a good size of the xsec.
fig, ax = plt.subplots(figsize=(13, 6))

# --- Adjust the right edge of teh axes to allow space for the legend's xbox
fig.subplots_adjust(right=0.75)

# --- phi and psi levels for contourin heads and stream lines
dphi, dpsi = 0.1, 0.1
phi_levels = np.arange(np.floor(xsec.hpp), np.ceil(xsec.stage), dphi)
psi_levels = np.arange(np.floor(psi.min()), np.ceil(psi.max()), dpsi)

# --- Titles and labels require phi and dpsi
fig.suptitle(xsec.name)
ttl = f", hartlijn ARK ={xsec.xy_hart_ARK[0]}"
ax.set(title=f"Stijghoogten, stroomlijnen en drukoverschot [dPhi={dphi} m dPsi={dpsi} m2/d]" + ttl,
       xlabel='x van hartlijn ARK',
       ylabel='z [NAP]')
ax.grid()

# --- Contour heads and streamlines
cs_phi = ax.contour(gr.xm, gr.zm, out['Phi'][:, 0, :], colors='b', linewidths=0.5, levels=phi_levels)
cs_psi = ax.contour(gr.x[1:-1], gr.z, psi, colors='r', linewidths=0.5, levels=psi_levels)

h_lbl_pts = find_horizonal_intersections(cs_phi, z0=-35)
v_lbl_pts = find_vertical_intersections(cs_phi, x0=0)
lbl_pts = h_lbl_pts + v_lbl_pts
ax.clabel(cs_phi, fmt='%.2f', manual=lbl_pts, inline=False, fontsize=8)

#ax.clabel(cs_phi, levels=cs_phi.levels, fontsize=10, fmt='%.2f')
# --- Color the background geologically
# --- Get the color of each cell
arr_RGB = xsec.leg_color_array(gr.idx_arr) / 255.

# --- Use pcolormesh to color the cells. gr.idx_arr is here just a dummy for it's shape
pcm = ax.pcolormesh(gr.X[:,0,:], gr.Z_corners[:,0,:], gr.idx_arr, shading='flat', alpha=1.0)

# --- Wipe out the data array (gr.idx_arr) inside pcolormesh, as we'll set facecolor manually next.
pcm.set_array(None)

# --- Set the facecolor manually
pcm.set_facecolor(arr_RGB.reshape(-1, 3))

# --- Add ARK water body and sheet pilings as patches
pARK = extent_patch(gr.ARK_extent, fc='blue', ec='none', alpha=1)
pDamwL = extent_patch(gr.damwL_extent, fc='k', ec='k', alpha=1)
pDamwR = extent_patch(gr.damwR_extent, fc='k', ec='k', alpha=1)

# --- Add these patches to our current axes.
ax.add_patch(pARK)
ax.add_patch(pDamwL)
ax.add_patch(pDamwR)

# --- Add head just below the resitance layer
phi7 = out['Phi'][np.where(gr.zm <= -7)[0][0], 0, :]
ax.plot(gr.xm, phi7, '--', color='k', lw=1)

# --- Adjust xlim for best view
ax.set_xlim(-250, 250)
# ax.set_aspect(50)

# --- Generate legend items/handles using color and properties of xsec.geo_units
handles = [
    Patch(facecolor=gu['color'], edgecolor='none',
          label=f"{k}: kh={gu['kh']:.1f}, kv={gu['kv']:.2f}, rhow={gu['rhow']:.0f}")
    for k, gu in xsec.geo_units.items()
]

# --- Generate legend using these handgles and put the bbox to anchor
leg_geo = ax.legend(handles=handles, loc='center left',
          bbox_to_anchor=(1, 0.5),
          title='Soil properties')
ax.add_artist(leg_geo)

# --- Generate a legend for the contoured heads and streamlines
leg_contours = ax.legend(
    handles =[
        Line2D([0], [0], color='k', lw=1.0, linestyle='--', label=r'$\phi$[z=-7m]'),
        Line2D([0], [0], color='b', lw=0.5, label='stijghoogte'),
        Line2D([0], [0], color='r', lw=0.5, label='stroomlijn')
    ],
    bbox_to_anchor=(1, 1),
    loc='upper left',
    title=r"Contours en $\phi$[z=-7]"
)
ax.add_artist(leg_contours)


# --- Add logo to reference the picture in the future
logo(fig, NOTEBOOK_NAME)

# --- Save the figure for reporting.
fig.savefig(os.path.join(dirs.images, f"ARK_XS{xsec.xRD}{xsec.yRD}.pdf"))

plt.show()

/var/folders/90/m51x_b713y561gzh2kzy18d00000gq/T/ipykernel_1197/4162686176.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [199]:
xsec.xARKmid_orig

np.float64(0.0)

# Wellen

In [233]:
# --- gravity and water densityWet density
g = 9.82 # N/kg
rho_wat = 1000.

# --- Compute head at the bottom of the cells
Phi_bot = out['Phi'].copy()  # --- at cell centers
Qz = out['Qz'].copy()        # --- flow in upward z-direction (at bottom of cells)
kv = gr.kv.copy()            # --- Vertical conductivity

# --- Special cells
Qz[kv[1:] == 0] = 0                  # --- Vertical flow at horizontal cell boundaries
kv += 1e-6                           # --- Cells insie sheet piling have kv=0
dPhi = Qz * gr.DZ[1:] / 2 / kv[1:]   # --- correction for shift to bettom of cell.
Phi_bot[:-1] += dPhi                 # --- Head ad cell bottom

# --- Water pressure head at the bottom of cells
h_wat = (Phi_bot - gr.Z[1:])

# --- Total pressure caused by overlying wet density at bottom of cells gr.Z[1:].
h_wet = np.cumsum(gr.rhow / rho_wat * gr.DZ, axis=0)

# --- Pressure head difference (not used)
delta_h = h_wet - h_wat
delta_h = delta_h.clip(-2, 2)
delta_h[xsec.gr.mask_ARK] = np.nan

# --- Pressure head ratio (used)
h_ratio = h_wat / h_wet
h_ratio[xsec.gr.mask_ARK] = np.nan # --- NaN inside the ARK water body

# --- Contour the 3 criteria levels hr
cs = plt.contour(gr.XM[:,0,:], gr.Z[1:,0,:], h_ratio[:, 0, :],
            levels=[0.8, 1.0, 1.2],
            colors=['g','orange','r'],
            linestyles='-.',
            linewidths=1)

# --- Legend for the ratio contours
leg_ratio = ax.legend(handles=[
    Line2D([0], [0], ls='-.', color='red',    lw=1, label=r"$\sigma_{wat}/\sigma{tot}$=0.8"),
    Line2D([0], [0], ls='-.', color='orange', lw=1, label=r"$\sigma_{wat}/\sigma{tot}$=1.0"),
    Line2D([0], [0], ls='-.', color='green',  lw=1, label=r"$\sigma_{wat}/\sigma{tot}$=1.2"),
    ],
    bbox_to_anchor=(1, 0),
    loc='lower left',
    title='Water overdruk')
ax.add_artist(leg_ratio)

ax.set_xlim(-250, 250)

# --- Save the figure for reporting.
fig.savefig(os.path.join(dirs.images, f"ARK_XS{xsec.xRD}{xsec.yRD}_cBot={xsec.cBot:.0f}d.pdf"))

plt.show()


/var/folders/90/m51x_b713y561gzh2kzy18d00000gq/T/ipykernel_1197/3854935713.py:28: RuntimeWarning: divide by zero encountered in divide
  h_ratio = h_wat / h_wet
/var/folders/90/m51x_b713y561gzh2kzy18d00000gq/T/ipykernel_1197/3854935713.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
